# Buril — entrenamiento del limpiador de trazos (Google Colab)

Cuaderno **driver fino**: no copia la lógica, sólo llama a los scripts del repo.
Así, cuando `tools/buril.py` o el dataset cambien, este cuaderno ya los usa en el
próximo `git pull`. La fuente de verdad son los `tools/*.py`.

Cadena: configurar versiones → clonar repo → **cache del `.npz`** en la release
`dataset-buril` (si no está, lo prepara y lo sube) → entrenar → publicar el
modelo en la release `modelo-buril` (el buzón que lee `android-apk.yml`).

**Versionado**: la celda 1 pide la versión del `.npz` y la del modelo (por
defecto ambas `v1.0.0-experimental.1a`). La versión del `.npz` va al nombre del
asset en `dataset-buril`; la del modelo va al manifiesto y a la release.

**Antes de empezar**: guardá en *Colab Secrets* (icono de llave 🔑, barra
izquierda) el secreto `GH_TOKEN` con un PAT fine-grained de permiso
*Contents: Read and write* sobre este repo. No lo escribas en una celda.

> Si estás probando este cuaderno **antes de mergear el PR**, cambiá `RAMA` en la
> celda 1 por el nombre de tu rama (Colab resuelve `main` por defecto).


In [ ]:
# @title 1. Configuración, clonado y dependencias
REPO = "LuisAlejandro544/Dibujo-"           # @param {type:"string"}
RAMA = "main"                               # @param {type:"string"}
VERSION_DATOS  = "v1.0.0-experimental.1a"   # @param {type:"string"}
VERSION_MODELO = "v1.0.0-experimental.1a"   # @param {type:"string"}
TAG_DATOS  = "dataset-buril"                # @param {type:"string"}
TAG_MODELO = "modelo-buril"                 # @param {type:"string"}
LADO          = 512                         # @param {type:"integer"}
POR_CATEGORIA = 200                         # @param {type:"integer"}
EPOCAS        = 60                          # @param {type:"integer"}
AUMENTOS      = 3                           # @param {type:"integer"}
PARCHES       = 16                          # @param {type:"integer"}
BANDA         = 1                           # @param {type:"integer"}
LR            = 0.001                       # @param {type:"number"}

import os, sys, subprocess

RUTA = "/content/Dibujo-"
if os.path.isdir(os.path.join(RUTA, ".git")):
    subprocess.run(["git", "fetch", "origin", RAMA], cwd=RUTA, check=True)
    subprocess.run(["git", "checkout", RAMA], cwd=RUTA, check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", RAMA], cwd=RUTA, check=True)
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", RAMA,
         f"https://github.com/{REPO}.git", RUTA], check=True)

os.chdir(RUTA)

# Usamos SIEMPRE el mismo intérprete del kernel (sys.executable) para que las
# dependencias instaladas y los scripts corran en el mismo entorno.
try:
    import tensorflow as tf
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "numpy", "tensorflow"], check=True)
    import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
print("Repositorio listo en", RUTA, "| rama", RAMA)
print("TensorFlow", tf.__version__, "| GPU:", [g.name for g in gpus] or "ninguna (CPU)")


In [ ]:
# @title 2. Utilidades de GitHub (release como buzón)
import json, os, requests
from google.colab import userdata

TOKEN = userdata.get("GH_TOKEN")
if not TOKEN:
    raise SystemExit("Falta el secreto GH_TOKEN en Colab (permiso Contents: Read and write).")

CABECERAS = {"Authorization": f"Bearer {TOKEN}", "Accept": "application/vnd.github+json"}
API = f"https://api.github.com/repos/{REPO}"


def asegurar_release(tag, nombre, cuerpo):
    """Devuelve la release [tag] creándola si no existe."""
    r = requests.get(f"{API}/releases/tags/{tag}", headers=CABECERAS)
    if r.status_code == 200:
        return r.json()
    r = requests.post(f"{API}/releases", headers=CABECERAS,
                      json={"tag_name": tag, "name": nombre, "body": cuerpo})
    r.raise_for_status()
    return r.json()


def bajar_asset(tag, nombre, destino):
    """Baja el asset [nombre] de la release [tag] a [destino]; True si existía."""
    r = requests.get(f"{API}/releases/tags/{tag}", headers=CABECERAS)
    if r.status_code != 200:
        return False
    for a in r.json().get("assets", []):
        if a["name"] == nombre:
            datos = requests.get(
                a["url"], headers={**CABECERAS, "Accept": "application/octet-stream"})
            datos.raise_for_status()
            os.makedirs(os.path.dirname(destino), exist_ok=True)
            with open(destino, "wb") as f:
                f.write(datos.content)
            print(f"Bajado {nombre} ({len(datos.content)} bytes) de {tag}")
            return True
    return False


def subir_asset(tag, ruta, nombre):
    """Sube (o reemplaza) [ruta] como asset [nombre] de la release [tag]."""
    release = asegurar_release(tag, tag, f"Buzón `{tag}`.")
    for viejo in requests.get(f"{API}/releases/{release['id']}/assets",
                              headers=CABECERAS).json():
        if viejo["name"] == nombre:
            requests.delete(f"{API}/releases/assets/{viejo['id']}", headers=CABECERAS)
    with open(ruta, "rb") as f:
        datos = f.read()
    subida = requests.post(
        f"https://uploads.github.com/repos/{REPO}/releases/{release['id']}/assets",
        headers={**CABECERAS, "Content-Type": "application/octet-stream"},
        params={"name": nombre}, data=datos)
    subida.raise_for_status()
    print(f"Publicado {nombre} ({len(datos)} bytes) en la release {tag}")


def actualizar_release(tag, nombre, cuerpo):
    """Fija el nombre y las notas de la release [tag]."""
    release = asegurar_release(tag, nombre, cuerpo)
    r = requests.patch(f"{API}/releases/{release['id']}", headers=CABECERAS,
                       json={"name": nombre, "body": cuerpo})
    r.raise_for_status()
    print(f"Release {tag} actualizada: {nombre}")


In [ ]:
# @title 3. Dataset .npz: recuperar de la release o prepararlo
DIR = ".cache/limpiar_trazos_planb/procesado"
NPZ = f"{DIR}/pares_esq_lado{LADO}.npz"
MANIF = f"{DIR}/manifiesto_esq_lado{LADO}.json"
ASSET_NPZ = f"pares_esq_lado{LADO}_pc{POR_CATEGORIA}_{VERSION_DATOS}.npz"
ASSET_MANIF = f"manifiesto_esq_lado{LADO}_pc{POR_CATEGORIA}_{VERSION_DATOS}.json"

asegurar_release(TAG_DATOS, "Dataset Buril (cache del .npz)",
                 "Cache del dataset comercial (Quick, Draw!, CC BY 4.0) que usa "
                 "`tools/buril.py`. Lo llena `tools/buril_colab.ipynb` para no "
                 "volver a bajar/preprocesar en cada sesión.")

if bajar_asset(TAG_DATOS, ASSET_NPZ, NPZ):
    bajar_asset(TAG_DATOS, ASSET_MANIF, MANIF)
    print(f"Dataset recuperado de la release {TAG_DATOS} ({VERSION_DATOS}).")
else:
    print("No hay cache para esta versión; preparando el dataset (puede tardar)...")
    subprocess.run([sys.executable,
                    "tools/preparar_dataset_limpiar_trazos_planb.py", "todo",
                    "--lado", str(LADO), "--por-categoria", str(POR_CATEGORIA),
                    "--version", VERSION_DATOS], check=True)
    subir_asset(TAG_DATOS, NPZ, ASSET_NPZ)
    if os.path.exists(MANIF):
        subir_asset(TAG_DATOS, MANIF, ASSET_MANIF)

print("npz listo:", NPZ, os.path.getsize(NPZ), "bytes")


In [ ]:
# @title 4. Entrenar Buril
from os.path import basename, splitext

SALIDA = ".cache/modelos/limpiar_trazos.tflite"

subprocess.run([sys.executable, "tools/buril.py",
                "--pares", NPZ,
                "--salida", SALIDA,
                "--epocas", str(EPOCAS),
                "--aumentos", str(AUMENTOS),
                "--banda", str(BANDA),
                "--lr", str(LR),
                "--parches-por-imagen", str(PARCHES),
                "--nombre", "Buril",
                "--version", VERSION_MODELO], check=True)

print("Modelo entrenado:", SALIDA, os.path.getsize(SALIDA), "bytes")


In [ ]:
# @title 5. Publicar el modelo en la release `modelo-buril`
CUERPO = "\n\n".join([
    f"Modelo **Buril {VERSION_MODELO}** (`limpiar_trazos.tflite`, limpiador de "
    "trazos float16) entrenado en Google Colab con `tools/buril_colab.ipynb`.",
    f"- Dataset: `{ASSET_NPZ}` (release `{TAG_DATOS}`).",
    f"- Épocas: {EPOCAS} · aumentos: {AUMENTOS} · parches/hoja: {PARCHES}.",
    f"- Objetivo: banda de {2 * BANDA + 1} px · LR inicial: {LR} (coseno) · pérdida BCE + Dice + clDice.",
    "- `android-apk.yml` baja este asset y lo inyecta en el APK.",
])
actualizar_release(TAG_MODELO, f"Modelo Buril {VERSION_MODELO}", CUERPO)

for archivo in [SALIDA, splitext(SALIDA)[0] + ".json"]:
    if os.path.exists(archivo):
        subir_asset(TAG_MODELO, archivo, basename(archivo))

with open(splitext(SALIDA)[0] + ".json", encoding="utf-8") as f:
    print(json.dumps(json.load(f), ensure_ascii=False, indent=2))


## Siguiente paso

La release `modelo-buril` ya tiene el `.tflite` de la versión pedida y el `.npz`
quedó cacheado en `dataset-buril`. El workflow `android-apk.yml` baja el `.tflite`
y lo inyecta en `app/src/main/assets/modelos/` antes de `assembleDebug`. Para
armar el APK con este modelo, corré *Actions → "APK Android (assembleDebug)" →
Run workflow*. Si la release está vacía, el APK se arma igual y la app cae al
adelgazado algorítmico.
